# QCHAN v4.0.0 — reviewed analytical preflight

**Family:** channel/device spectral effects  
**Run role:** controlled analytical preflight only; cohort extraction and freezing are disabled.

QCHAN asks how the gain-normalized long-term speech spectrum differs from a frozen, task-matched, subject-balanced leave-one-subject-out reference. The four outputs are a profile, not a scalar:

1. reference-relative LTAS distance;
2. rolloff-95 deficit;
3. high-band-ratio deficit;
4. spectral-tilt steepening.

The family measures cohort-relative spectral deviation and attenuation proxies. It does not identify a microphone/device, recover a transfer function, or separate acquisition effects from speech phenotype. Human QC, diagnosis, ALSFRS-R, and prediction are not analytical-validation gates.

## 0. Environment, controls, and candidate output contract

In [1]:
from __future__ import annotations

from dataclasses import replace
from datetime import datetime, timezone
from pathlib import Path
from tempfile import TemporaryDirectory
import hashlib
import json
import math
import os
import shutil
import subprocess
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
from scipy import signal, stats
from IPython.display import Markdown, display


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "paper1_qc").exists():
            return candidate
    raise FileNotFoundError("Open this notebook from inside the paper1 repository.")


ROOT = find_project_root()
for source_root in [ROOT / "src reviewed", ROOT / "src"]:
    if str(source_root) not in sys.path:
        sys.path.insert(0, str(source_root))

from paper1_qc_reviewed.qchan_v400 import (
    ANALYSIS_FEATURES,
    DEFAULT_PARAMETERS,
    FEATURE_DEFINITIONS,
    MEASUREMENT_VERSION,
    PREFLIGHT_REVISION,
    TimeInterval,
    apply_gain_db,
    broad_notch_filter,
    build_subject_balanced_loso_references,
    compute_reference_relative_features,
    extract_controlled_features,
    extract_recording_spectrum,
    feature_registry_frame,
    full_span_interval,
    lowpass_filter,
    reference_from_recording_spectrum,
    reference_vintage_sha256,
    smooth_high_shelf,
    synthetic_speech_like,
)

warnings.filterwarnings("default")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

RUN_PACKAGE_TESTS = True
RUN_CODEC_ROUNDTRIP = True
RUN_COHORT_EXTRACTION = False
PUBLISH_AND_FREEZE = False
SCIENTIFIC_REVIEW_DECISION = "PENDING"

if RUN_COHORT_EXTRACTION:
    raise RuntimeError("Cohort extraction is prohibited in the QCHAN preflight notebook.")
if PUBLISH_AND_FREEZE:
    raise RuntimeError("Freezing is prohibited in the QCHAN preflight notebook.")

FS = DEFAULT_PARAMETERS.analysis_sample_rate_hz
CANDIDATE = ROOT / "outputs reviewed" / "channel_device" / "qchan-v4.0.0-candidate"
TABLES = CANDIDATE / "tables"
FIGURES = CANDIDATE / "figures"
MANIFESTS = CANDIDATE / "manifests"
for directory in [TABLES, FIGURES, MANIFESTS]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project:", ROOT)
print("Measurement version:", MEASUREMENT_VERSION)
print("Preflight revision:", PREFLIGHT_REVISION)
print("Candidate outputs:", CANDIDATE)
print("Cohort extraction enabled:", RUN_COHORT_EXTRACTION)
print("Freeze enabled:", PUBLISH_AND_FREEZE)

Project: C:\Users\musikicn\Desktop\Nevena_project\Paper_1\paper_1
Measurement version: qchan-v4.0.0
Preflight revision: qchan-v4.0.0-preflight-r1
Candidate outputs: C:\Users\musikicn\Desktop\Nevena_project\Paper_1\paper_1\outputs reviewed\channel_device\qchan-v4.0.0-candidate
Cohort extraction enabled: False
Freeze enabled: False


## 1. Helpers, artifact contract, and package tests

In [2]:
def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def save_table(frame: pd.DataFrame, stem: str):
    csv_path = TABLES / f"{stem}.csv"
    frame.to_csv(csv_path, index=False)
    try:
        frame.to_parquet(TABLES / f"{stem}.parquet", index=False)
    except Exception as exc:
        print(f"Parquet not written for {stem}: {type(exc).__name__}: {exc}")
    return csv_path


def save_figure_bundle(fig, stem: str, source: pd.DataFrame, caption: str, provenance: dict):
    png = FIGURES / f"{stem}.png"
    svg = FIGURES / f"{stem}.svg"
    pdf = FIGURES / f"{stem}.pdf"
    source_csv = FIGURES / f"{stem}.source.csv"
    caption_path = FIGURES / f"{stem}.caption.md"
    provenance_path = FIGURES / f"{stem}.provenance.json"
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    source.to_csv(source_csv, index=False)
    caption_path.write_text(caption, encoding="utf-8")
    provenance_path.write_text(json.dumps(json_safe(provenance), indent=2), encoding="utf-8")
    return {
        "stem": stem,
        "png": str(png.relative_to(CANDIDATE)),
        "svg": str(svg.relative_to(CANDIDATE)),
        "pdf": str(pdf.relative_to(CANDIDATE)),
        "source_csv": str(source_csv.relative_to(CANDIDATE)),
        "caption": str(caption_path.relative_to(CANDIDATE)),
        "provenance": str(provenance_path.relative_to(CANDIDATE)),
    }


def feature_row(waveform, reference, name, source_rate=None, parameters=DEFAULT_PARAMETERS):
    return extract_controlled_features(
        np.asarray(waveform, dtype=np.float64),
        reference=reference,
        logical_recording_id=name,
        source_sample_rate_hz=source_rate,
        parameters=parameters,
    )


def add_noise_at_snr(signal_waveform, noise_waveform, snr_db):
    x = np.asarray(signal_waveform, dtype=np.float64)
    n = np.asarray(noise_waveform, dtype=np.float64)
    x_rms = float(np.sqrt(np.mean(x * x)))
    n_rms = float(np.sqrt(np.mean(n * n)))
    scale = x_rms / max(n_rms, 1e-15) / (10 ** (snr_db / 20.0))
    return x + n * scale


def roundtrip_source_rate(waveform, source_rate):
    x = np.asarray(waveform, dtype=np.float64)
    if source_rate == FS:
        return x.copy()
    gcd1 = math.gcd(FS, int(source_rate))
    down = signal.resample_poly(x, int(source_rate) // gcd1, FS // gcd1)
    gcd2 = math.gcd(int(source_rate), FS)
    up = signal.resample_poly(down, FS // gcd2, int(source_rate) // gcd2)
    if len(up) < len(x):
        up = np.pad(up, (0, len(x) - len(up)))
    return np.asarray(up[:len(x)], dtype=np.float64)


package_tests_passed = False
package_test_output = "not run"
if RUN_PACKAGE_TESTS:
    completed = subprocess.run(
        [
            str(ROOT / ".venv" / "Scripts" / "python.exe") if os.name == "nt" else sys.executable,
            "-m", "pytest", str(ROOT / "tests reviewed" / "test_qchan_v400.py"),
            "-q", "--disable-warnings",
        ],
        cwd=ROOT,
        check=False,
        capture_output=True,
        text=True,
    )
    package_test_output = completed.stdout + completed.stderr
    print(package_test_output)
    package_tests_passed = completed.returncode == 0
    if not package_tests_passed:
        raise RuntimeError("Reviewed QCHAN package tests failed.")

registry = feature_registry_frame()
save_table(registry, "qchan_v400_feature_registry")
parameters = pd.DataFrame([
    {"parameter": key, "value": value}
    for key, value in DEFAULT_PARAMETERS.to_dict().items()
])
save_table(parameters, "qchan_v400_parameters")
display(registry)

................                                                         [100%]



,name,display_name,subdomain,role,unit,estimand,orientation,claim_boundary,minimum_support,known_confounds,evidence_class
0,qchan_ltas_distance_db,Reference-relative LTAS distance,spectral coloration,primary nonordinal,dB RMS,RMS difference between target and frozen-refer...,Higher means greater spectral deviation; not i...,Cohort-relative spectral deviation; not device...,Target: at least 3 s guarded strict speech; re...,"ALS phenotype, phonetic composition, sex, age,...",study-specific reference-relative estimator
1,qchan_rolloff95_deficit_hz,Reference-relative rolloff95 deficit,bandwidth attenuation,primary,Hz,"max(0, reference rolloff95 - target rolloff95)",Higher means less upper spectral extent than t...,One-sided bandwidth-loss proxy; not a codec or...,Same target/reference support as LTAS distance...,"Fricative content, dysarthria, additive noise,...",study-specific reference-relative estimator
2,qchan_highband_ratio_deficit,Reference-relative high-band deficit,bandwidth attenuation,secondary,proportion,"max(0, reference minus target) for 3-7.5-kHz /...",Higher means less relative high-band energy.,Secondary attenuation proxy; content-dependent...,Same target/reference support as LTAS distance.,"Frication, articulation, additive high-frequen...",study-specific reference-relative estimator
3,qchan_tilt_steepening_db_per_oct,Reference-relative spectral-tilt steepening,spectral coloration,secondary phenotype-sensitive,dB/octave,"max(0, reference Theil-Sen log-LTAS slope - ta...",Higher means a steeper downward spectral tilt ...,Secondary phenotype-sensitive proxy; not indep...,Same target/reference support as LTAS distance.,"Glottal source, vocal effort, dysarthria, sex,...",study-specific reference-relative estimator


## 2. Controlled reference and transformation contract (G1–G3)

In [3]:
baseline_waveform = synthetic_speech_like(duration_sec=12.0, sample_rate_hz=FS)
baseline_spectrum = extract_recording_spectrum(
    baseline_waveform,
    FS,
    strict_speech=full_span_interval(baseline_waveform, FS),
    logical_recording_id="controlled_baseline",
)
controlled_reference = reference_from_recording_spectrum(baseline_spectrum)
baseline_features = feature_row(baseline_waveform, controlled_reference, "baseline")

transformation_rows = []
variants = {
    "baseline": baseline_waveform,
    "gain_-12_db": apply_gain_db(baseline_waveform, -12.0),
    "gain_+9_db": apply_gain_db(baseline_waveform, 9.0),
    "polarity": -baseline_waveform,
    "dc_+0.25": baseline_waveform + 0.25,
}
for condition, waveform in variants.items():
    row = feature_row(waveform, controlled_reference, condition)
    transformation_rows.append({"condition": condition, **{feature: row[feature] for feature in ANALYSIS_FEATURES}})

# Common time shift with a shifted interval.
pad_sec = 0.75
shifted = np.pad(baseline_waveform, (int(pad_sec * FS), 0))
shift_interval = [TimeInterval(
    pad_sec - DEFAULT_PARAMETERS.speech_boundary_guard_ms / 1000.0,
    pad_sec + len(baseline_waveform) / FS + DEFAULT_PARAMETERS.speech_boundary_guard_ms / 1000.0,
)]
shift_spectrum = extract_recording_spectrum(
    shifted, FS, strict_speech=shift_interval, logical_recording_id="common_time_shift"
)
shift_row = compute_reference_relative_features(shift_spectrum, controlled_reference)
transformation_rows.append({"condition": "common_time_shift", **{feature: shift_row[feature] for feature in ANALYSIS_FEATURES}})
transformation_controls = pd.DataFrame(transformation_rows)
save_table(transformation_controls, "qchan_v400_transformation_controls")

baseline_vector = transformation_controls.loc[transformation_controls.condition.eq("baseline"), list(ANALYSIS_FEATURES)].iloc[0]
transformation_delta = transformation_controls.copy()
for feature in ANALYSIS_FEATURES:
    transformation_delta[f"{feature}_absolute_delta"] = (transformation_delta[feature] - baseline_vector[feature]).abs()
max_invariance_error = float(transformation_delta.filter(like="_absolute_delta").to_numpy().max())

source_rate_rows = []
for source_rate in [48000, 24000, 16000, 12000, 8000]:
    waveform = roundtrip_source_rate(baseline_waveform, source_rate)
    row = feature_row(waveform, controlled_reference, f"source_rate_{source_rate}", source_rate=source_rate)
    source_rate_rows.append({
        "source_sample_rate_hz": source_rate,
        "source_nyquist_hz": source_rate / 2,
        "source_bandwidth_limited": row["qchan_source_bandwidth_limited"],
        **{feature: row[feature] for feature in ANALYSIS_FEATURES},
    })
source_rate_characterization = pd.DataFrame(source_rate_rows)
save_table(source_rate_characterization, "qchan_v400_source_rate_characterization")

codec_rows = []
codec_errors = []
if RUN_CODEC_ROUNDTRIP:
    ffmpeg = shutil.which("ffmpeg")
    if not ffmpeg:
        raise RuntimeError("ffmpeg is required for codec characterization.")
    with TemporaryDirectory() as temporary_directory:
        temporary = Path(temporary_directory)
        source_wav = temporary / "source.wav"
        sf.write(source_wav, baseline_waveform.astype(np.float32), FS, subtype="PCM_24")
        conditions = [
            ("flac", ["-c:a", "flac"], ".flac"),
            ("opus_64k", ["-c:a", "libopus", "-b:a", "64k"], ".opus"),
            ("aac_96k", ["-c:a", "aac", "-b:a", "96k"], ".m4a"),
        ]
        for name, args, extension in conditions:
            encoded = temporary / f"encoded{extension}"
            decoded = temporary / f"decoded_{name}.wav"
            try:
                subprocess.run([ffmpeg, "-y", "-v", "error", "-i", str(source_wav), *args, str(encoded)], check=True)
                subprocess.run([ffmpeg, "-y", "-v", "error", "-i", str(encoded), "-ar", str(FS), "-ac", "1", str(decoded)], check=True)
                waveform, sample_rate = sf.read(decoded, dtype="float64")
                waveform = np.asarray(waveform, dtype=np.float64)[:len(baseline_waveform)]
                if len(waveform) < len(baseline_waveform):
                    waveform = np.pad(waveform, (0, len(baseline_waveform) - len(waveform)))
                row = feature_row(waveform, controlled_reference, name, source_rate=sample_rate)
                codec_rows.append({"condition": name, **{feature: row[feature] for feature in ANALYSIS_FEATURES}})
            except Exception as exc:
                codec_errors.append({"condition": name, "error": f"{type(exc).__name__}: {exc}"})
codec_characterization = pd.DataFrame(codec_rows)
codec_error_table = pd.DataFrame(codec_errors, columns=["condition", "error"])
save_table(codec_characterization, "qchan_v400_codec_characterization")
save_table(codec_error_table, "qchan_v400_codec_errors")

display(transformation_controls)
display(source_rate_characterization)
display(codec_characterization)

,condition,qchan_ltas_distance_db,qchan_rolloff95_deficit_hz,qchan_highband_ratio_deficit,qchan_tilt_steepening_db_per_oct
0,baseline,0.000000e+00,0.0,0.0,0.0
1,gain_-12_db,2.823411e-15,0.0,0.0,0.0
2,gain_+9_db,3.645007e-15,0.0,0.0,0.0
3,polarity,0.000000e+00,0.0,0.0,0.0
4,dc_+0.25,0.000000e+00,0.0,0.0,0.0
5,common_time_shift,0.000000e+00,0.0,0.0,0.0


,source_sample_rate_hz,source_nyquist_hz,source_bandwidth_limited,qchan_ltas_distance_db,qchan_rolloff95_deficit_hz,qchan_highband_ratio_deficit,qchan_tilt_steepening_db_per_oct
0,48000,24000.0,False,0.093656,147.386810,0.008736,0.000000
1,24000,12000.0,False,0.093671,147.382238,0.008738,0.000000
2,16000,8000.0,False,0.000000,0.000000,0.000000,0.000000
3,12000,6000.0,True,2.745882,1737.145536,0.133384,0.000000
4,8000,4000.0,True,18.252529,3529.666265,0.402791,0.010952


,condition,qchan_ltas_distance_db,qchan_rolloff95_deficit_hz,qchan_highband_ratio_deficit,qchan_tilt_steepening_db_per_oct
0,flac,0.000027,0.000000,0.000000,0.0
1,opus_64k,0.084984,123.953197,0.007911,0.0
2,aac_96k,0.019962,0.000000,0.000000,0.0


## 3. Construct response and discriminant controls (G4–G5)

In [4]:
lowpass_rows = []
for cutoff_hz in [7500, 6500, 5500, 4800, 4500, 4000, 3400]:
    transformed = lowpass_filter(baseline_waveform, FS, cutoff_hz)
    row = feature_row(transformed, controlled_reference, f"lowpass_{cutoff_hz}")
    lowpass_rows.append({"condition": "lowpass", "dose": cutoff_hz, "dose_label": f"{cutoff_hz} Hz", **{feature: row[feature] for feature in ANALYSIS_FEATURES}})
lowpass_dose = pd.DataFrame(lowpass_rows)
save_table(lowpass_dose, "qchan_v400_lowpass_dose")

shelf_rows = []
for gain_db in [-12, -6, 0, 6, 12]:
    transformed = smooth_high_shelf(baseline_waveform, FS, gain_db)
    row = feature_row(transformed, controlled_reference, f"shelf_{gain_db}")
    shelf_rows.append({"condition": "high_shelf", "dose": gain_db, "dose_label": f"{gain_db:+d} dB", **{feature: row[feature] for feature in ANALYSIS_FEATURES}})
shelf_dose = pd.DataFrame(shelf_rows)
save_table(shelf_dose, "qchan_v400_shelf_dose")

notch_rows = []
for depth_db in [0, -6, -12, -18, -24]:
    transformed = broad_notch_filter(baseline_waveform, FS, depth_db=depth_db)
    row = feature_row(transformed, controlled_reference, f"notch_{depth_db}")
    notch_rows.append({"condition": "broad_notch", "dose": depth_db, "dose_label": f"{depth_db:d} dB", **{feature: row[feature] for feature in ANALYSIS_FEATURES}})
notch_dose = pd.DataFrame(notch_rows)
save_table(notch_dose, "qchan_v400_notch_dose")

rng = np.random.default_rng(20260803)
white = rng.normal(size=len(baseline_waveform))
hf_sos = signal.butter(4, [4500/(FS/2), 7400/(FS/2)], btype="band", output="sos")
lf_sos = signal.butter(4, [100/(FS/2), 800/(FS/2)], btype="band", output="sos")
high_noise = signal.sosfilt(hf_sos, white)
low_noise = signal.sosfilt(lf_sos, white)
lowpassed = lowpass_filter(baseline_waveform, FS, 3400)

control_signals = {
    "dry_baseline": baseline_waveform,
    "lowpass_3400": lowpassed,
    "lowpass_3400_plus_hf_noise_20dB": add_noise_at_snr(lowpassed, high_noise, 20),
    "dry_plus_hf_noise_20dB": add_noise_at_snr(baseline_waveform, high_noise, 20),
    "dry_plus_lf_noise_20dB": add_noise_at_snr(baseline_waveform, low_noise, 20),
    "high_shelf_+12dB": smooth_high_shelf(baseline_waveform, FS, 12),
    "high_shelf_-12dB": smooth_high_shelf(baseline_waveform, FS, -12),
    "broad_notch_-18dB": broad_notch_filter(baseline_waveform, FS, depth_db=-18),
}
discriminant_rows = []
for name, waveform in control_signals.items():
    row = feature_row(waveform, controlled_reference, name)
    discriminant_rows.append({"condition": name, **{feature: row[feature] for feature in ANALYSIS_FEATURES},
                              "rolloff_signed_difference_hz": row["qchan_rolloff95_signed_difference_hz"],
                              "highband_signed_difference": row["qchan_highband_ratio_signed_difference"],
                              "tilt_signed_difference_db_per_oct": row["qchan_tilt_signed_difference_db_per_oct"]})
discriminant_controls = pd.DataFrame(discriminant_rows)
save_table(discriminant_controls, "qchan_v400_discriminant_controls")

display(lowpass_dose)
display(shelf_dose)
display(discriminant_controls)

,condition,dose,dose_label,qchan_ltas_distance_db,qchan_rolloff95_deficit_hz,qchan_highband_ratio_deficit,qchan_tilt_steepening_db_per_oct
0,lowpass,7500,7500 Hz,0.022775,35.916121,0.002028,0.000000e+00
1,lowpass,6500,6500 Hz,0.898274,1023.604879,0.067535,8.894286e-07
2,lowpass,5500,5500 Hz,6.349145,1999.861914,0.159630,9.401274e-07
3,lowpass,4800,4800 Hz,15.767543,2675.340138,0.245383,1.022864e-07
4,lowpass,4500,4500 Hz,18.095690,2951.596833,0.290854,0.000000e+00
5,lowpass,4000,4000 Hz,19.765716,3429.149365,0.383480,1.752062e-02
6,lowpass,3400,3400 Hz,24.289758,3976.022225,0.525307,3.431845e-02


,condition,dose,dose_label,qchan_ltas_distance_db,qchan_rolloff95_deficit_hz,qchan_highband_ratio_deficit,qchan_tilt_steepening_db_per_oct
0,high_shelf,-12,-12 dB,5.011868e+00,1431.174305,0.469944,0.199391
1,high_shelf,-6,-6 dB,2.888589e+00,330.729166,0.265897,0.167941
2,high_shelf,0,+0 dB,2.305305e-15,0.000000,0.000000,0.000000
3,high_shelf,6,+6 dB,3.834087e+00,0.000000,0.000000,0.000000
4,high_shelf,12,+12 dB,8.279281e+00,0.000000,0.000000,0.000000


,condition,qchan_ltas_distance_db,qchan_rolloff95_deficit_hz,qchan_highband_ratio_deficit,qchan_tilt_steepening_db_per_oct,rolloff_signed_difference_hz,highband_signed_difference,tilt_signed_difference_db_per_oct
0,dry_baseline,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,lowpass_3400,24.289758,3976.022225,0.525307,0.034318,3976.022225,0.525307,0.034318
2,lowpass_3400_plus_hf_noise_20dB,7.447644,3933.361753,0.516264,0.034375,3933.361753,0.516264,0.034375
3,dry_plus_hf_noise_20dB,0.046237,0.000000,0.000000,0.000005,-3.354065,-0.003997,0.000005
4,dry_plus_lf_noise_20dB,0.277447,3.773216,0.006342,0.125431,3.773216,0.006342,0.125431
5,high_shelf_+12dB,8.279281,0.000000,0.000000,0.000000,-145.408971,-0.298918,-0.279694
6,high_shelf_-12dB,5.011868,1431.174305,0.469944,0.199391,1431.174305,0.469944,0.199391
7,broad_notch_-18dB,2.576552,0.000000,0.000000,0.050992,-25.856659,-0.048176,0.050992


## 4. Support, floor, boundary, and reference preflight (G6)

In [5]:
support_rows = []
for duration_sec in [2.0, 2.9, 3.0, 5.0, 10.0]:
    waveform = synthetic_speech_like(duration_sec=duration_sec, sample_rate_hz=FS, seed=20260730)
    spectrum = extract_recording_spectrum(
        waveform, FS, strict_speech=full_span_interval(waveform, FS),
        logical_recording_id=f"duration_{duration_sec}",
    )
    support_rows.append({
        "duration_sec": duration_sec,
        "status": spectrum.status,
        "support_tier": spectrum.support_tier,
        "guarded_speech_support_sec": spectrum.guarded_speech_support_sec,
        "valid_frame_count": spectrum.valid_frame_count,
    })
support_preflight = pd.DataFrame(support_rows)
save_table(support_preflight, "qchan_v400_support_preflight")

parameter_rows = []
for frame_ms, hop_ms, guard_ms in [(30, 10, 200), (40, 10, 200), (50, 10, 200), (40, 10, 100), (40, 10, 300)]:
    params = replace(DEFAULT_PARAMETERS, frame_ms=frame_ms, hop_ms=hop_ms, speech_boundary_guard_ms=guard_ms)
    baseline_spec = extract_recording_spectrum(
        baseline_waveform, FS, strict_speech=full_span_interval(baseline_waveform, FS),
        logical_recording_id=f"baseline_{frame_ms}_{guard_ms}", parameters=params,
    )
    ref = reference_from_recording_spectrum(baseline_spec, parameters=params)
    transformed = lowpass_filter(baseline_waveform, FS, 4000)
    obs = extract_recording_spectrum(
        transformed, FS, strict_speech=full_span_interval(transformed, FS),
        logical_recording_id=f"lp_{frame_ms}_{guard_ms}", parameters=params,
    )
    row = compute_reference_relative_features(obs, ref, parameters=params)
    parameter_rows.append({"frame_ms": frame_ms, "hop_ms": hop_ms, "guard_ms": guard_ms, **{feature: row[feature] for feature in ANALYSIS_FEATURES}})
parameter_sensitivity = pd.DataFrame(parameter_rows)
save_table(parameter_sensitivity, "qchan_v400_frame_guard_sensitivity")

floor_rows = []
for floor_db in [-100, -90, -80, -70, -60]:
    params = replace(DEFAULT_PARAMETERS, relative_psd_floor_db=floor_db)
    baseline_spec = extract_recording_spectrum(
        baseline_waveform, FS, strict_speech=full_span_interval(baseline_waveform, FS),
        logical_recording_id=f"baseline_floor_{floor_db}", parameters=params,
    )
    ref = reference_from_recording_spectrum(baseline_spec, parameters=params)
    for cutoff_hz in [6500, 5500, 4500, 4000, 3400]:
        transformed = lowpass_filter(baseline_waveform, FS, cutoff_hz)
        obs = extract_recording_spectrum(
            transformed, FS, strict_speech=full_span_interval(transformed, FS),
            logical_recording_id=f"floor_{floor_db}_lp_{cutoff_hz}", parameters=params,
        )
        row = compute_reference_relative_features(obs, ref, parameters=params)
        floor_rows.append({"relative_psd_floor_db": floor_db, "cutoff_hz": cutoff_hz, **{feature: row[feature] for feature in ANALYSIS_FEATURES}})
floor_sensitivity = pd.DataFrame(floor_rows)
save_table(floor_sensitivity, "qchan_v400_floor_sensitivity")

# Synthetic subject-balanced LOSO reference contract.
reference_spectra = {}
reference_meta_rows = []
for subject_index in range(6):
    for recording_index in range(2):
        waveform = synthetic_speech_like(duration_sec=6.0, sample_rate_hz=FS, seed=1000 + subject_index)
        waveform = smooth_high_shelf(waveform, FS, (subject_index - 2.5) * 0.4)
        recording_id = f"subject_{subject_index}_recording_{recording_index}"
        spectrum = extract_recording_spectrum(
            waveform, FS, strict_speech=full_span_interval(waveform, FS),
            logical_recording_id=recording_id,
        )
        reference_spectra[recording_id] = spectrum
        reference_meta_rows.append({"logical_recording_id": recording_id, "subject_id": f"subject_{subject_index}", "task_stratum": "bamboo"})
reference_metadata = pd.DataFrame(reference_meta_rows)
references = build_subject_balanced_loso_references(reference_spectra, reference_metadata)
reference_rows = []
for recording_id, reference in references.items():
    target_subject = reference_metadata.set_index("logical_recording_id").loc[recording_id, "subject_id"]
    reference_rows.append({
        "logical_recording_id": recording_id,
        "target_subject_id": target_subject,
        "reference_status": reference.status,
        "reference_recording_count": reference.recording_count,
        "reference_subject_count": reference.subject_count,
        "target_subject_excluded": target_subject not in reference.member_subject_ids,
        "reference_key": reference.reference_key,
        "reference_sha256": reference.reference_sha256,
        "reference_vintage_sha256": reference.reference_vintage_sha256,
    })
reference_contract = pd.DataFrame(reference_rows)
save_table(reference_contract, "qchan_v400_reference_contract")

# Insufficient reference has no fallback.
limited_metadata = reference_metadata.loc[reference_metadata.subject_id.isin(["subject_0", "subject_1", "subject_2", "subject_3", "subject_4"])].copy()
limited_spectra = {recording_id: reference_spectra[recording_id] for recording_id in limited_metadata.logical_recording_id}
limited_references = build_subject_balanced_loso_references(limited_spectra, limited_metadata)
no_fallback_ok = all(reference.status == "reference_unavailable" for reference in limited_references.values())

# Reference vintage changes when membership changes.
vintage_full = reference_vintage_sha256(reference_spectra, reference_metadata)
reduced_metadata = reference_metadata.iloc[:-1].copy()
vintage_reduced = reference_vintage_sha256({recording_id: reference_spectra[recording_id] for recording_id in reduced_metadata.logical_recording_id}, reduced_metadata)

# Floor rank stability across low-pass doses.
floor_rank_rows = []
for feature in ANALYSIS_FEATURES:
    pivot = floor_sensitivity.pivot(index="cutoff_hz", columns="relative_psd_floor_db", values=feature)
    for left in pivot.columns:
        for right in pivot.columns:
            if left < right:
                rho = stats.spearmanr(pivot[left], pivot[right]).statistic
                floor_rank_rows.append({"feature": feature, "floor_left_db": left, "floor_right_db": right, "spearman_rho": rho})
floor_rank_stability = pd.DataFrame(floor_rank_rows)
save_table(floor_rank_stability, "qchan_v400_floor_rank_stability")

display(support_preflight)
display(parameter_sensitivity)
display(reference_contract.head())
display(floor_rank_stability.groupby("feature").spearman_rho.min())

,duration_sec,status,support_tier,guarded_speech_support_sec,valid_frame_count
0,2.0,insufficient_strict_speech_support,unavailable,2.0,197
1,2.9,insufficient_strict_speech_support,unavailable,2.9,287
2,3.0,measured,minimum,3.0,297
3,5.0,measured,moderate,5.0,497
4,10.0,measured,high,10.0,997


,frame_ms,hop_ms,guard_ms,qchan_ltas_distance_db,qchan_rolloff95_deficit_hz,qchan_highband_ratio_deficit,qchan_tilt_steepening_db_per_oct
0,30,10,200,19.782959,3428.453764,0.383483,0.011242
1,40,10,200,19.765716,3429.149365,0.383480,0.017521
2,50,10,200,19.748920,3429.554935,0.383493,0.017102
3,40,10,100,19.765716,3429.149365,0.383480,0.017521
4,40,10,300,19.769002,3430.512542,0.383336,0.010679


,logical_recording_id,target_subject_id,reference_status,reference_recording_count,reference_subject_count,target_subject_excluded,reference_key,reference_sha256,reference_vintage_sha256
0,subject_0_recording_0,subject_0,measured,10,5,True,48bed1314c66569116cb,a68ffb66badc0d737bd59b5d4a677f43b93ef127acf2db...,5cfc0cfa9c3ef7587f70f0e0d0d7b7ce9a075cc2434770...
1,subject_0_recording_1,subject_0,measured,10,5,True,48bed1314c66569116cb,a68ffb66badc0d737bd59b5d4a677f43b93ef127acf2db...,5cfc0cfa9c3ef7587f70f0e0d0d7b7ce9a075cc2434770...
2,subject_1_recording_0,subject_1,measured,10,5,True,1c98ca2f772138226f09,fa374996222e8d102e92c0bff594a32efed3c8cbd43b0e...,5cfc0cfa9c3ef7587f70f0e0d0d7b7ce9a075cc2434770...
3,subject_1_recording_1,subject_1,measured,10,5,True,1c98ca2f772138226f09,fa374996222e8d102e92c0bff594a32efed3c8cbd43b0e...,5cfc0cfa9c3ef7587f70f0e0d0d7b7ce9a075cc2434770...
4,subject_2_recording_0,subject_2,measured,10,5,True,09acf23e637ca464c14b,01dee75c6621fe8860787ef188487f1033f90f710dd044...,5cfc0cfa9c3ef7587f70f0e0d0d7b7ce9a075cc2434770...


feature
qchan_highband_ratio_deficit        1.0
qchan_ltas_distance_db              1.0
qchan_rolloff95_deficit_hz          1.0
qchan_tilt_steepening_db_per_oct    1.0
Name: spearman_rho, dtype: float64

## 5. Standardized preflight figures A–C

In [6]:
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.22})
figure_index_rows = []

# Panel A — construct response.
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, feature, label in [
    (axes[0, 0], "qchan_ltas_distance_db", "LTAS distance (dB RMS)"),
    (axes[0, 1], "qchan_rolloff95_deficit_hz", "Rolloff-95 deficit (Hz)"),
    (axes[1, 0], "qchan_highband_ratio_deficit", "High-band deficit (proportion)"),
    (axes[1, 1], "qchan_tilt_steepening_db_per_oct", "Tilt steepening (dB/octave)"),
]:
    ax.plot(lowpass_dose["dose"], lowpass_dose[feature], marker="o")
    ax.invert_xaxis()
    ax.set(xlabel="Low-pass cutoff (Hz; decreasing dose)", ylabel=label, title=label)
fig.suptitle("Panel A — controlled bandwidth-restriction response")
fig.tight_layout()
figure_index_rows.append(save_figure_bundle(
    fig,
    "A_construct_response",
    pd.concat([lowpass_dose, shelf_dose, notch_dose], ignore_index=True, sort=False),
    "Panel A. Controlled QCHAN response to decreasing low-pass cutoff. Companion source data also retain two-sided high-shelf and broad-notch perturbations. LTAS distance is nonordinal and responds to both attenuation and enhancement; the one-sided deficits are expected to respond primarily to attenuation.",
    {"panel": "A", "measurement_version": MEASUREMENT_VERSION, "preflight_revision": PREFLIGHT_REVISION, "parameters": DEFAULT_PARAMETERS.to_dict(), "source_tables": ["qchan_v400_lowpass_dose.csv", "qchan_v400_shelf_dose.csv", "qchan_v400_notch_dose.csv"]},
))
plt.close(fig)

# Panel B — discriminant specificity.
plot_features = list(ANALYSIS_FEATURES)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feature in zip(axes.ravel(), plot_features):
    local = discriminant_controls[["condition", feature]].copy()
    ax.barh(local["condition"], local[feature])
    ax.set(title=FEATURE_DEFINITIONS[feature]["display_name"], xlabel=FEATURE_DEFINITIONS[feature]["unit"])
fig.suptitle("Panel B — competing mechanisms and non-identifiability controls")
fig.tight_layout()
figure_index_rows.append(save_figure_bundle(
    fig,
    "B_discriminant_specificity",
    discriminant_controls,
    "Panel B. QCHAN responses to bandwidth restriction, high- and low-frequency additive noise, two-sided shelves, and a broad notch. High-frequency noise can mask one-sided bandwidth deficits, while speech-spectrum coloration without bandwidth restriction can increase LTAS distance. These controls define QCHAN as a channel-risk proxy rather than a device or transfer-function measurement.",
    {"panel": "B", "measurement_version": MEASUREMENT_VERSION, "preflight_revision": PREFLIGHT_REVISION, "source_table": "qchan_v400_discriminant_controls.csv"},
))
plt.close(fig)

# Panel C — transformation, source rate, and codec contract.
fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))
max_delta_by_condition = transformation_delta.set_index("condition").filter(like="_absolute_delta").max(axis=1)
axes[0].bar(max_delta_by_condition.index, max_delta_by_condition.values)
axes[0].tick_params(axis="x", rotation=45)
axes[0].set(title="Gain/polarity/DC/time-shift invariance", ylabel="Maximum absolute feature delta")
axes[1].plot(source_rate_characterization["source_sample_rate_hz"], source_rate_characterization["qchan_rolloff95_deficit_hz"], marker="o", label="Rolloff deficit")
axes[1].set(title="Source-rate roundtrip", xlabel="Native/source sample rate (Hz)", ylabel="Rolloff-95 deficit (Hz)")
if len(codec_characterization):
    axes[2].bar(codec_characterization["condition"], codec_characterization["qchan_ltas_distance_db"])
    axes[2].tick_params(axis="x", rotation=35)
    axes[2].set(title="Codec characterization", ylabel="LTAS distance (dB RMS)")
else:
    axes[2].text(0.5, 0.5, "Codec characterization unavailable", ha="center", va="center")
    axes[2].set_axis_off()
fig.suptitle("Panel C — transformation and analysis-view contract")
fig.tight_layout()
panel_c_source = pd.concat([
    transformation_delta.assign(source="transformation"),
    source_rate_characterization.assign(source="source_rate"),
    codec_characterization.assign(source="codec"),
], ignore_index=True, sort=False)
figure_index_rows.append(save_figure_bundle(
    fig,
    "C_transformation_contract",
    panel_c_source,
    "Panel C. Gain, polarity, constant DC, and common time shifts are invariant under the reviewed analysis contract. Source-rate roundtrips and lossy codecs are characterized rather than assumed invariant because bandwidth and spectral shaping are part of the QCHAN construct.",
    {"panel": "C", "measurement_version": MEASUREMENT_VERSION, "preflight_revision": PREFLIGHT_REVISION, "source_tables": ["qchan_v400_transformation_controls.csv", "qchan_v400_source_rate_characterization.csv", "qchan_v400_codec_characterization.csv"]},
))
plt.close(fig)

figure_index = pd.DataFrame(figure_index_rows)
save_table(figure_index, "qchan_v400_preflight_figure_index")
display(figure_index)

,stem,png,svg,pdf,source_csv,caption,provenance
0,A_construct_response,figures\A_construct_response.png,figures\A_construct_response.svg,figures\A_construct_response.pdf,figures\A_construct_response.source.csv,figures\A_construct_response.caption.md,figures\A_construct_response.provenance.json
1,B_discriminant_specificity,figures\B_discriminant_specificity.png,figures\B_discriminant_specificity.svg,figures\B_discriminant_specificity.pdf,figures\B_discriminant_specificity.source.csv,figures\B_discriminant_specificity.caption.md,figures\B_discriminant_specificity.provenance....
2,C_transformation_contract,figures\C_transformation_contract.png,figures\C_transformation_contract.svg,figures\C_transformation_contract.pdf,figures\C_transformation_contract.source.csv,figures\C_transformation_contract.caption.md,figures\C_transformation_contract.provenance.json


## 6. G1–G6 checks, gate summary, and candidate manifest

In [7]:
lowpass_order = {
    feature: bool(lowpass_dose.sort_values("dose", ascending=False)[feature].is_monotonic_increasing)
    for feature in ["qchan_ltas_distance_db", "qchan_rolloff95_deficit_hz", "qchan_highband_ratio_deficit"]
}
negative_shelf = shelf_dose.loc[shelf_dose.dose.eq(-12)].iloc[0]
positive_shelf = shelf_dose.loc[shelf_dose.dose.eq(12)].iloc[0]
plain_lp = discriminant_controls.set_index("condition").loc["lowpass_3400"]
masked_lp = discriminant_controls.set_index("condition").loc["lowpass_3400_plus_hf_noise_20dB"]
minimum_floor_rho = float(floor_rank_stability.spearman_rho.min())
codec_complete = bool((not RUN_CODEC_ROUNDTRIP) or (len(codec_characterization) == 3 and codec_error_table.empty))

checks = pd.DataFrame([
    {"gate": "G1", "check": "exact four-feature registry and no scalar", "passed": len(registry) == 4 and tuple(registry.name) == ANALYSIS_FEATURES, "observed": list(registry.name), "required": list(ANALYSIS_FEATURES), "note": "feature-first profile", "blocking": True},
    {"gate": "G1", "check": "analysis-view and native-bandwidth contract declared", "passed": DEFAULT_PARAMETERS.analysis_sample_rate_hz == 16000 and DEFAULT_PARAMETERS.analysis_high_hz == 7500, "observed": DEFAULT_PARAMETERS.to_dict(), "required": "16 kHz analysis view; native rate retained", "note": "no equalization or amplitude normalization", "blocking": True},
    {"gate": "G1", "check": "reference identity includes membership and parameters", "passed": vintage_full != vintage_reduced and reference_contract.reference_sha256.ne("").all(), "observed": {"full": vintage_full, "reduced": vintage_reduced}, "required": "membership-sensitive hashes", "note": "reference vintage is feature identity", "blocking": True},
    {"gate": "G2", "check": "package numerical tests pass", "passed": package_tests_passed, "observed": package_test_output.splitlines()[-1] if package_test_output else "", "required": "all pass", "note": "reviewed module tests", "blocking": True},
    {"gate": "G2", "check": "controlled baseline is exact zero against itself", "passed": all(abs(float(baseline_features[feature])) <= 1e-12 for feature in ANALYSIS_FEATURES), "observed": {feature: baseline_features[feature] for feature in ANALYSIS_FEATURES}, "required": "all <=1e-12", "note": "exact reference identity", "blocking": True},
    {"gate": "G2", "check": "one-sided zeros retain signed precursors", "passed": bool(positive_shelf.qchan_rolloff95_deficit_hz == 0 and positive_shelf.qchan_highband_ratio_deficit == 0), "observed": {"rolloff_deficit": positive_shelf.qchan_rolloff95_deficit_hz, "highband_deficit": positive_shelf.qchan_highband_ratio_deficit}, "required": "zero deficit for upward deviations; signed audit retained", "note": "zero is measured one-sided truncation, not missingness", "blocking": True},
    {"gate": "G3", "check": "gain polarity DC and common-shift invariance", "passed": max_invariance_error <= 1e-9, "observed": max_invariance_error, "required": "<=1e-9", "note": "shape features", "blocking": True},
    {"gate": "G3", "check": "source-rate bandwidth behavior is characterized", "passed": bool(
        source_rate_characterization.loc[source_rate_characterization.source_sample_rate_hz.eq(8000), "qchan_rolloff95_deficit_hz"].iloc[0]
        > source_rate_characterization.loc[source_rate_characterization.source_sample_rate_hz.eq(12000), "qchan_rolloff95_deficit_hz"].iloc[0]
        > source_rate_characterization.loc[source_rate_characterization.source_sample_rate_hz.ge(16000), "qchan_rolloff95_deficit_hz"].max()
        and source_rate_characterization.loc[source_rate_characterization.source_sample_rate_hz.lt(15000), "source_bandwidth_limited"].astype(bool).all()
        and ~source_rate_characterization.loc[source_rate_characterization.source_sample_rate_hz.ge(16000), "source_bandwidth_limited"].astype(bool).any()
    ), "observed": source_rate_characterization[["source_sample_rate_hz", "source_bandwidth_limited", "qchan_rolloff95_deficit_hz"]].to_dict(orient="records"), "required": "8-kHz deficit > 12-kHz deficit > maximum full-band roundtrip deficit; native limitation flags correct", "note": "full-band resampling is characterized, not required to be exact; sub-15-kHz source rate is a bandwidth limitation", "blocking": True},
    {"gate": "G3", "check": "codec roundtrips characterized without errors", "passed": codec_complete, "observed": {"rows": len(codec_characterization), "errors": len(codec_error_table)}, "required": "3 rows and 0 errors when enabled", "note": "lossy codecs may move QCHAN", "blocking": True},
    {"gate": "G4", "check": "low-pass dose orders LTAS distance", "passed": lowpass_order["qchan_ltas_distance_db"], "observed": lowpass_dose[["dose", "qchan_ltas_distance_db"]].to_dict(orient="records"), "required": "nondecreasing as cutoff falls", "note": "controlled positive control", "blocking": True},
    {"gate": "G4", "check": "low-pass dose orders rolloff and high-band deficits", "passed": lowpass_order["qchan_rolloff95_deficit_hz"] and lowpass_order["qchan_highband_ratio_deficit"], "observed": lowpass_order, "required": "both ordered", "note": "bandwidth attenuation", "blocking": True},
    {"gate": "G4", "check": "two-sided shelves and notch move nonordinal LTAS distance", "passed": bool(negative_shelf.qchan_ltas_distance_db > 1 and positive_shelf.qchan_ltas_distance_db > 1 and notch_dose.loc[notch_dose.dose.eq(-18), "qchan_ltas_distance_db"].iloc[0] > 1), "observed": {"negative_shelf": negative_shelf.qchan_ltas_distance_db, "positive_shelf": positive_shelf.qchan_ltas_distance_db}, "required": ">1 dB RMS for both shelf signs and notch", "note": "LTAS distance is nonordinal", "blocking": True},
    {"gate": "G5", "check": "high-frequency noise masking is demonstrated", "passed": bool(masked_lp.qchan_highband_ratio_deficit < plain_lp.qchan_highband_ratio_deficit and masked_lp.qchan_rolloff95_deficit_hz < plain_lp.qchan_rolloff95_deficit_hz), "observed": {"plain_highband": plain_lp.qchan_highband_ratio_deficit, "masked_highband": masked_lp.qchan_highband_ratio_deficit, "plain_rolloff": plain_lp.qchan_rolloff95_deficit_hz, "masked_rolloff": masked_lp.qchan_rolloff95_deficit_hz}, "required": "masking demonstrated", "note": "one-sided deficits are not source-specific", "blocking": True},
    {"gate": "G5", "check": "coloration without bandwidth loss is demonstrated", "passed": bool(positive_shelf.qchan_ltas_distance_db > 1 and positive_shelf.qchan_rolloff95_deficit_hz == 0), "observed": {"ltas": positive_shelf.qchan_ltas_distance_db, "rolloff_deficit": positive_shelf.qchan_rolloff95_deficit_hz}, "required": "LTAS distance positive with zero attenuation deficit", "note": "subdomains remain separate", "blocking": True},
    {"gate": "G5", "check": "speech phenotype and additive-noise non-identifiability remains explicit", "passed": True, "observed": "conditional claim retained", "required": "no device or transfer-function claim", "note": "G5 final status is CONDITIONAL", "blocking": False},
    {"gate": "G6", "check": "minimum target support transitions at 3 seconds", "passed": bool(support_preflight.loc[support_preflight.duration_sec.lt(3), "status"].ne("measured").all() and support_preflight.loc[support_preflight.duration_sec.ge(3), "status"].eq("measured").all()), "observed": support_preflight.to_dict(orient="records"), "required": "<3 s unavailable; >=3 s measured", "note": "support is not quality", "blocking": True},
    {"gate": "G6", "check": "LOSO references exclude target and are subject-balanced", "passed": bool(reference_contract.target_subject_excluded.all() and reference_contract.reference_subject_count.eq(5).all()), "observed": reference_contract[["target_subject_excluded", "reference_subject_count", "reference_recording_count"]].drop_duplicates().to_dict(orient="records"), "required": "target excluded; 5 other subjects", "note": "within-subject median then across-subject median", "blocking": True},
    {"gate": "G6", "check": "insufficient reference support has no fallback", "passed": no_fallback_ok, "observed": no_fallback_ok, "required": True, "note": "no global or cross-task fallback", "blocking": True},
    {"gate": "G6", "check": "floor sensitivity preserves controlled dose rankings", "passed": minimum_floor_rho >= 0.90, "observed": minimum_floor_rho, "required": ">=0.90", "note": "absolute LTAS-distance scale remains floor-dependent", "blocking": True},
    {"gate": "G6", "check": "frame and guard variants remain finite", "passed": bool(parameter_sensitivity[list(ANALYSIS_FEATURES)].apply(np.isfinite).all().all()), "observed": len(parameter_sensitivity), "required": "all finite", "note": "cohort ranking sensitivity remains pending", "blocking": True},
])

save_table(checks, "qchan_v400_preflight_all_checks")
blocking_pass = bool(checks.loc[checks.blocking.astype(bool), "passed"].astype(bool).all())

gate_summary = pd.DataFrame([
    {"gate": "G1", "status": "PASS" if checks.loc[checks.gate.eq("G1") & checks.blocking, "passed"].all() else "FAIL", "scope": "contract and provenance"},
    {"gate": "G2", "status": "PASS" if checks.loc[checks.gate.eq("G2") & checks.blocking, "passed"].all() else "FAIL", "scope": "numerical correctness"},
    {"gate": "G3", "status": "PASS" if checks.loc[checks.gate.eq("G3") & checks.blocking, "passed"].all() else "FAIL", "scope": "transformation behavior"},
    {"gate": "G4", "status": "PASS" if checks.loc[checks.gate.eq("G4") & checks.blocking, "passed"].all() else "FAIL", "scope": "construct response"},
    {"gate": "G5", "status": "CONDITIONAL" if checks.loc[checks.gate.eq("G5") & checks.blocking, "passed"].all() else "FAIL", "scope": "discriminant validity; irreducible non-identifiability retained"},
    {"gate": "G6", "status": "PREFLIGHT_PASS" if checks.loc[checks.gate.eq("G6") & checks.blocking, "passed"].all() else "FAIL", "scope": "support, parameter, and reference preflight"},
    {"gate": "G7", "status": "PENDING", "scope": "corrected cohort evidence"},
    {"gate": "G8", "status": "PENDING", "scope": "persistence, redundancy, participant balancing"},
    {"gate": "G9", "status": "N/A", "scope": "no retained event detector"},
    {"gate": "G10", "status": "PENDING", "scope": "feature decisions and immutable freeze"},
])
save_table(gate_summary, "qchan_v400_gate_summary")

manifest = {
    "measurement_version": MEASUREMENT_VERSION,
    "preflight_revision": PREFLIGHT_REVISION,
    "candidate_only": True,
    "preflight_blocking_checks_pass": blocking_pass,
    "package_tests_passed": package_tests_passed,
    "cohort_extraction_completed": False,
    "freeze_allowed": False,
    "scientific_review_decision": SCIENTIFIC_REVIEW_DECISION,
    "analysis_features": list(ANALYSIS_FEATURES),
    "parameters": DEFAULT_PARAMETERS.to_dict(),
    "figure_panels_complete": ["A", "B", "C"],
    "remaining_panels": ["D", "E", "F", "G", "H", "J"],
    "panel_i_status": "N/A_no_retained_event_detector",
    "family_scalar_constructed": False,
    "standalone_gate_allowed": False,
    "reference_rule": "task-matched subject-balanced leave-one-subject-out; no fallback",
    "claim_boundary": "cohort-relative spectral deviation and attenuation proxies; no device identification or pure transfer function",
    "created_utc": datetime.now(timezone.utc).isoformat(),
}
(MANIFESTS / "qchan_v400_preflight_manifest.json").write_text(json.dumps(json_safe(manifest), indent=2), encoding="utf-8")

print("QCHAN v4.0.0 REVIEWED PREFLIGHT COMPLETE")
print(json.dumps(manifest, indent=2))
display(checks)
display(gate_summary)
if not blocking_pass:
    raise RuntimeError("QCHAN preflight blocking checks failed.")

Parquet not written for qchan_v400_preflight_all_checks: ArrowInvalid: ('cannot mix list and non-list, non-null values', 'Conversion failed for column observed with type object')
QCHAN v4.0.0 REVIEWED PREFLIGHT COMPLETE
{
  "measurement_version": "qchan-v4.0.0",
  "preflight_revision": "qchan-v4.0.0-preflight-r1",
  "candidate_only": true,
  "preflight_blocking_checks_pass": true,
  "package_tests_passed": true,
  "cohort_extraction_completed": false,
  "freeze_allowed": false,
  "scientific_review_decision": "PENDING",
  "analysis_features": [
    "qchan_ltas_distance_db",
    "qchan_rolloff95_deficit_hz",
    "qchan_highband_ratio_deficit",
    "qchan_tilt_steepening_db_per_oct"
  ],
  "parameters": {
    "analysis_sample_rate_hz": 16000,
    "frame_ms": 40.0,
    "hop_ms": 10.0,
    "speech_boundary_guard_ms": 200.0,
    "n_fft": 2048,
    "analysis_low_hz": 100.0,
    "analysis_high_hz": 7500.0,
    "highband_low_hz": 3000.0,
    "highband_high_hz": 7500.0,
    "tilt_low_hz": 100.0

,gate,check,passed,observed,required,note,blocking
0,G1,exact four-feature registry and no scalar,True,"[qchan_ltas_distance_db, qchan_rolloff95_defic...","[qchan_ltas_distance_db, qchan_rolloff95_defic...",feature-first profile,True
1,G1,analysis-view and native-bandwidth contract de...,True,"{'analysis_sample_rate_hz': 16000, 'frame_ms':...",16 kHz analysis view; native rate retained,no equalization or amplitude normalization,True
2,G1,reference identity includes membership and par...,True,{'full': '5cfc0cfa9c3ef7587f70f0e0d0d7b7ce9a07...,membership-sensitive hashes,reference vintage is feature identity,True
3,G2,package numerical tests pass,True,[32m.[0m[32m.[0m[32m.[0m[32m.[0m[32m....,all pass,reviewed module tests,True
4,G2,controlled baseline is exact zero against itself,True,"{'qchan_ltas_distance_db': 0.0, 'qchan_rolloff...",all <=1e-12,exact reference identity,True
5,G2,one-sided zeros retain signed precursors,True,"{'rolloff_deficit': 0.0, 'highband_deficit': 0.0}",zero deficit for upward deviations; signed aud...,"zero is measured one-sided truncation, not mis...",True
6,G3,gain polarity DC and common-shift invariance,True,0.0,<=1e-9,shape features,True
7,G3,source-rate bandwidth behavior is characterized,True,"[{'source_sample_rate_hz': 48000, 'source_band...",8-kHz deficit > 12-kHz deficit > maximum full-...,"full-band resampling is characterized, not req...",True
8,G3,codec roundtrips characterized without errors,True,"{'rows': 3, 'errors': 0}",3 rows and 0 errors when enabled,lossy codecs may move QCHAN,True
9,G4,low-pass dose orders LTAS distance,True,"[{'dose': 7500, 'qchan_ltas_distance_db': 0.02...",nondecreasing as cutoff falls,controlled positive control,True


,gate,status,scope
0,G1,PASS,contract and provenance
1,G2,PASS,numerical correctness
2,G3,PASS,transformation behavior
3,G4,PASS,construct response
4,G5,CONDITIONAL,discriminant validity; irreducible non-identif...
5,G6,PREFLIGHT_PASS,"support, parameter, and reference preflight"
6,G7,PENDING,corrected cohort evidence
7,G8,PENDING,"persistence, redundancy, participant balancing"
8,G9,N/A,no retained event detector
9,G10,PENDING,feature decisions and immutable freeze
